# Лаборатория 1. Первый бот и его первый провал

**Что мы сделаем:** напишем бота для сайта сервисного центра «Полярис» самым простым
способом — просто спросим модель. Посмотрим на трассу запроса, посчитаем цену, а потом
поймаем первый провал и заведём три файла, которые будем вести весь курс.

| Шаг | Что делаем | Кто пишет |
|---|---|---|
| 1 | Подключаемся к модели, смотрим на трассу запроса | дано |
| 2 | Правила бота в роли `system` | пишем вместе |
| 3 | Температура: почему ответ каждый раз разный | дано |
| 4 | Первый провал: бот отвечает про компанию | дано |
| 5 | Журнал случаев `cases.jsonl` | пишем вместе |
| 6 | Первая проверка: прогон по журналу | пишем вместе |
| 7 | Чиним просьбой «не выдумывай» и замеряем | пишем вместе |
| 8 | Сколько это стоит на тысяче вопросов | дано |
| 9 | Задания | пиши сам |

**Запросов к модели:** около 25. **Цена:** меньше цента на дешёвой модели — в шаге 8
посчитаем точно.

In [ ]:
!pip -q install openai

## Шаг 1. Подключаемся `[дано]`

Модель живёт не у нас: мы отправляем ей текст по сети и получаем ответ. Нужны два
значения — **адрес** сервиса и **ключ**.

Есть два варианта, и код одинаковый для обоих:

1. **Учебный доступ.** Адрес `https://ai9.adelfos.ru/api/v1`, ключ — код от преподавателя.
   Ключ настоящего провайдера лежит на сервере, у нас его нет.
2. **Свой ключ.** Заводится на openrouter.ai, адрес `https://openrouter.ai/api/v1`,
   ключ вида `sk-or-...`. Так вы ни от кого не зависите, но платите сами.

В Colab значения удобно хранить в секретах (значок ключа слева): `AI_BASE_URL`, `AI_KEY`.
Тогда они не попадут в код и не уедут вместе с ноутбуком — правило из первого дня работы
с любым API.

In [ ]:
import getpass
import json
import os
import time
from pprint import pprint

from openai import OpenAI


def iz_sekretov(imya, po_umolchaniyu=None):
    """Достаёт значение из секретов Colab, затем из переменных окружения."""
    try:
        from google.colab import userdata
        znachenie = userdata.get(imya)
        if znachenie:
            return znachenie
    except Exception:
        pass
    return os.environ.get(imya) or po_umolchaniyu


BASE_URL = iz_sekretov("AI_BASE_URL", "https://ai9.adelfos.ru/api/v1")
MODEL = iz_sekretov("AI_MODEL", "qwen/qwen3.7-flash")
API_KEY = iz_sekretov("AI_KEY")

# Сервер проверяет ключ, только когда мы к нему обращаемся. Поэтому делаем один лёгкий
# запрос — список моделей — и, если ключ не подошёл, спрашиваем его заново. Цикл
# крутится до тех пор, пока подключение не заработает: так ноутбук не падает
# посреди урока из-за опечатки в ключе.
# timeout: не ждать ответа вечно. max_retries=0: повторять будем сами и осознанно (модуль 2).
client = None
while client is None:
    if not API_KEY:
        API_KEY = getpass.getpass("Ключ или код доступа: ")
    probnyy = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=60, max_retries=0)
    try:
        probnyy.models.list()
        client = probnyy
        print(f"Подключились. Адрес: {BASE_URL}, модель: {MODEL}")
    except Exception as oshibka:
        print(f"Не подошло: {type(oshibka).__name__} — {str(oshibka)[:120]}")
        print("Проверьте ключ (и адрес, если он свой) и введите ключ заново.")
        API_KEY = None

### Трасса: видеть, что уходит и что приходит

Главная привычка курса — **смотреть на то, что реально отправлено**. Почти все странные
ответы объясняются не «модель плохая», а «мы отправили не то».

Функция `sprosit` ниже делает один запрос и умеет печатать трассу: сообщения, длину,
токены, время и цену. Дальше мы пользуемся ей везде.

In [ ]:
# Цены из прайса OpenRouter: доллары за миллион токенов. Для своей модели — поправьте.
CENA_VHODA, CENA_VYHODA = 0.03, 0.13


def cena(usage):
    return (usage.prompt_tokens * CENA_VHODA + usage.completion_tokens * CENA_VYHODA) / 1e6


def sprosit(soobshcheniya, trassa=False, max_tokens=300, temperature=0, **dop):
    """Один запрос к модели. Возвращает (текст, usage). trassa=True печатает подробности."""
    if trassa:
        print("→ ЗАПРОС")
        for s in soobshcheniya:
            tekst = s["content"].replace("\n", " ⏎ ")
            print(f"   [{s['role']:<9}] {len(tekst):>5} симв. | {tekst[:100]}{'…' if len(tekst) > 100 else ''}")
    nachalo = time.perf_counter()
    otvet = client.chat.completions.create(
        model=MODEL, temperature=temperature, max_tokens=max_tokens, messages=soobshcheniya, **dop)
    sekundy = time.perf_counter() - nachalo
    tekst = (otvet.choices[0].message.content or "").strip()
    if trassa:
        print("← ОТВЕТ")
        print(f"   модель: {otvet.model}, {sekundy:.1f} с, finish_reason={otvet.choices[0].finish_reason}")
        print(f"   токены: вход {otvet.usage.prompt_tokens}, выход {otvet.usage.completion_tokens}, "
              f"цена ${cena(otvet.usage):.6f}")
        print(f"   текст: {tekst}")
    return tekst, otvet.usage


tekst, usage = sprosit([{"role": "user", "content": "Привет! Ответь одним предложением, кто ты."}], trassa=True)

**Что посмотреть в выводе:**

1. **Сообщение всего одно** — с ролью `user`. Никаких правил мы пока не задали.
2. **Вход больше, чем кажется.** Мы отправили короткую фразу, а токенов на входе
   больше: к тексту добавляется служебная разметка ролей.
3. **Цена** — шестой знак после запятой. Пока это выглядит как «бесплатно», но в шаге 8
   мы умножим на тысячу вопросов.
4. **`finish_reason=stop`** значит «модель закончила сама». Другие значения появятся
   в модуле 2, когда ответ упрётся в ограничение длины.

## Шаг 2. Правила бота `[пишем вместе]`

Наш бот — не общий чат, а помощник конкретной компании. Правила кладут в сообщение
с ролью `system`. Сравним два запроса с одним и тем же вопросом.

In [ ]:
VOPROS = "Здравствуйте! Вы чините посудомоечные машины?"

print("=== без правил ===")
bez_pravil, _ = sprosit([{"role": "user", "content": VOPROS}])
print(bez_pravil[:400])

PRAVILA = (
    "Ты помощник сервисного центра «Полярис»: ремонт бытовой техники. "
    "Отвечай вежливо и коротко, максимум два предложения, на русском языке."
)

print("\n=== с правилами ===")
s_pravilami, _ = sprosit([
    {"role": "system", "content": PRAVILA},
    {"role": "user", "content": VOPROS},
])
print(s_pravilami[:400])

**Что посмотреть в выводе:** ответ без правил — общий и длинный, модель не знает, от чьего
имени говорит. С правилами он короче и «от лица компании».

Но заметьте главное: **правила не добавили модели знаний о «Полярисе»**. Она по-прежнему
не видела ни одной страницы сайта. Это и приведёт нас к провалу в шаге 4.

**Попробуйте сами прямо сейчас:** допишите в `PRAVILA` строку «Если не знаешь ответа,
скажи об этом прямо» и повторите запрос. Запомните, что получилось, — вернёмся к этому
в шаге 7.

## Шаг 3. Температура `[дано]`

Модель выбирает продолжение из нескольких правдоподобных. При `temperature=0` она берёт
самое вероятное, при высокой — рискует. Зададим один вопрос по три раза.

In [ ]:
VOPROS_TVORCH = "Придумай короткий слоган для сервисного центра «Полярис»."

for temperatura in (0, 1.2):
    print(f"--- temperature={temperatura} ---")
    for popytka in range(3):
        otvet, _ = sprosit(
            [{"role": "system", "content": PRAVILA}, {"role": "user", "content": VOPROS_TVORCH}],
            max_tokens=40, temperature=temperatura)
        print(f"   {popytka + 1}: {otvet}")

**Что посмотреть в выводе:**

1. При `temperature=0` три ответа почти одинаковые. Почти — потому что «ноль» не даёт
   полной гарантии: при подготовке лаборатории третий ответ отличался последними словами
   («будет работать как новая» → «прослужит ещё долго»). Полной повторяемости у модели нет.
2. При `1.2` ответы заметно разные — иногда удачнее, иногда странные.
3. Для справочного бота нужен предсказуемый ответ, поэтому дальше везде `temperature=0`.
   Для слоганов и текстов — наоборот.

И важная оговорка: `temperature=0` не значит «одинаково навсегда». Провайдер может
обновить модель, и ответ изменится. Поэтому проверки, которые мы напишем в шаге 6,
не сравнивают ответ дословно.

## Шаг 4. Первый провал `[дано]`

Теперь зададим боту настоящие вопросы гостей. Правильные ответы мы знаем: они написаны
на сайте «Поляриса». Вот выдержка из него — **боту мы её не показываем**, она нужна нам,
чтобы проверить его ответы.

In [ ]:
PRAVDA = {
    "subbota": "суббота: с 10:00 до 17:00",
    "garantiya": "гарантия на работы 12 месяцев, на запчасти 6 месяцев",
    "podshipniki": "замена подшипников стиральной машины — от 4900 ₽",
    "samokaty": "электросамокаты не ремонтируем: такой услуги на сайте нет",
    "diagnostika": "диагностика на дому 1200 ₽, бесплатно при согласии на ремонт",
}

VOPROSY_GOSTEY = [
    ("subbota", "Во сколько вы закрываетесь в субботу?"),
    ("garantiya", "Какая у вас гарантия на ремонт?"),
    ("podshipniki", "Сколько стоит замена подшипников в стиральной машине?"),
    ("samokaty", "Вы чините электросамокаты?"),
    ("diagnostika", "Диагностика платная?"),
]


def bot_v1(vopros, pravila=PRAVILA):
    """v1: просто спрашиваем модель. Никаких документов у неё нет."""
    return sprosit([{"role": "system", "content": pravila}, {"role": "user", "content": vopros}], max_tokens=120)[0]


print("Спрашиваем v1 то, что спрашивают настоящие гости:\n")
otvety_v1 = {}
for klyuch, vopros in VOPROSY_GOSTEY:
    otvet = bot_v1(vopros)
    otvety_v1[klyuch] = otvet
    print(f"❓ {vopros}")
    print(f"🤖 {otvet}")
    print(f"✅ на сайте: {PRAVDA[klyuch]}\n")

**Что посмотреть в выводе — и это главный момент всего модуля:**

1. Бот ответил на **все** вопросы. Ни разу не сказал «я не знаю, у меня нет данных».
2. Ответы звучат уверенно и по-деловому: часы работы, сроки гарантии, цены.
3. Сравните их со строкой «на сайте». Вот что вышло при подготовке лаборатории:

| Вопрос | Ответ v1 | На сайте |
|---|---|---|
| суббота | «работаем до 18:00» | до 17:00 |
| гарантия | «12 месяцев» | 12 месяцев — **угадала** |
| подшипники | «зависит от модели, уточните» | от 4900 ₽ |
| электросамокаты | «не занимаемся, обратитесь в профильный сервис» | верно, но знать этого она не могла |
| диагностика | «да, платная, зависит от техники» | 1200 ₽, бесплатно при ремонте |

Отдельно разберите две строки, которые выглядят «правильными».

**Гарантия.** Ответ совпал с сайтом — и это худший вид провала. Модель не знала срок,
она назвала типичный для отрасли. В другой компании гарантия три месяца, и ответ был бы
неверным при той же уверенности. Совпадение не значит знание.

**Электросамокаты.** Бот отказался — по сути верно. Но он не проверял список услуг,
а просто продолжил фразу про «сервис бытовой техники». Если завтра «Полярис» добавит
ремонт самокатов, бот продолжит отказывать.

Модель не врёт нарочно. Она делает ровно то, для чего сделана: продолжает текст
правдоподобным образом. На вопрос «во сколько закрываетесь в субботу» правдоподобное
продолжение — какое-нибудь время. Откуда ему взяться правильным, если страницу сайта
мы не показывали?

> **Выдумка (галлюцинация)** — уверенный правдоподобный ответ, не основанный на фактах.
> Это не сбой, а прямое следствие того, как устроена модель.

Опаснее всего здесь не сама ошибка, а **тон**. Бот нигде не написал «возможно» или
«уточните»: гость читает «до 18:00» и приезжает к закрытой двери.

## Шаг 5. Журнал случаев `[пишем вместе]`

Пойманный провал нельзя оставлять в голове: через неделю он забудется, а через месяц
вернётся. Заводим `cases.jsonl` — по одному случаю на строку.

Формат простой: что спросили, что ждём в ответе, чем плох текущий ответ.

In [ ]:
SLUCHAI = []


def zapisat_sluchay(klyuch, vopros, zhdem, chto_ne_tak):
    SLUCHAI.append({"id": klyuch, "vopros": vopros, "zhdem": zhdem,
                    "chto_ne_tak": chto_ne_tak, "poymano_v": "модуль 1"})


for klyuch, vopros in VOPROSY_GOSTEY:
    zhdem = {
        "subbota": "17:00",
        "garantiya": "12 месяц",
        "podshipniki": "4900",
        "samokaty": "не знаю",
        "diagnostika": "1200",
    }[klyuch]
    zapisat_sluchay(klyuch, vopros, zhdem, f"v1 ответила: {otvety_v1[klyuch][:60]}")

with open("cases.jsonl", "w") as f:
    for sluchay in SLUCHAI:
        f.write(json.dumps(sluchay, ensure_ascii=False) + "\n")

print(f"Записано случаев: {len(SLUCHAI)}\n")
print(open("cases.jsonl").read())

Обратите внимание на случай `samokaty`: ожидаемый ответ — **«не знаю»**. Правильный ответ
бота не всегда содержит факт; иногда правильно именно отказаться. Без таких случаев
в наборе легко сделать бота, который бойко отвечает на всё подряд и всегда неправ.

И ещё: мы ждём именно честное «не знаю», а не «нет, не чиним». Пока у бота нет списка
услуг, любое утверждение о них — угадывание, даже если угадал верно.

## Шаг 6. Первая проверка `[пишем вместе]`

Теперь превратим журнал в проверку. Сравнивать ответы дословно нельзя: модель каждый раз
формулирует иначе. Поэтому проверяем по существу:

* если ждём факт — он должен встретиться в ответе (число, срок);
* если ждём «не знаю» — в ответе должен быть отказ, а не обещание услуги.

In [ ]:
def proverit(otvet, zhdem):
    nizhniy = otvet.lower()
    if zhdem == "не знаю":
        otkaz = any(s in nizhniy for s in ("не зна", "не уточн", "нет информац", "не указан", "уточните"))
        obeshchanie = any(s in nizhniy for s in ("да,", "чиним", "ремонтируем", "конечно"))
        return otkaz and not obeshchanie
    return zhdem.lower() in nizhniy


def progon(bot, nazvanie):
    """Гоняет бота по всем случаям и печатает отчёт. Возвращает число верных ответов."""
    print(f"=== {nazvanie} ===")
    verno = 0
    for sluchay in SLUCHAI:
        otvet = bot(sluchay["vopros"])
        ok = proverit(otvet, sluchay["zhdem"])
        verno += ok
        print(f"{'✅' if ok else '❌'} {sluchay['id']:<12} ждём «{sluchay['zhdem']}» | {otvet[:70]}")
    print(f"Верных ответов: {verno} из {len(SLUCHAI)}\n")
    return verno


verno_v1 = progon(bot_v1, "v1: просто спрашиваем модель")

**Что посмотреть в выводе:** у нас появилось **число**. Не «бот вроде отвечает», а
«столько-то из пяти». При подготовке лаборатории вышло **1 из 5**, причём единственная
«зелёная» строка — та самая угаданная гарантия. То есть реально бот не знает ничего.

С этого числа начинается инженерия: дальше любое изменение сравнивается с ним.

Это ровно то, что делает `pytest`: набор случаев, для каждого известен правильный
результат, красное или зелёное. В модуле 7, когда код переедет в проект, этот прогон
станет обычным тестом (`shablony/tests/test_cases.py` — его заготовка).

## Шаг 7. Чиним просьбой — и проверяем `[пишем вместе]`

Первая мысль любого разработчика: «попрошу модель не выдумывать». Это одна строка,
поэтому попробовать обязательно нужно — но так же обязательно **замерить**.

In [ ]:
PRAVILA_STROGIE = (
    PRAVILA +
    " Отвечай только тем, что знаешь про этот сервисный центр наверняка. "
    "Если не знаешь точно, ответь «Не знаю, уточните у оператора». Не придумывай цены, "
    "сроки и услуги."
)


def bot_v1_1(vopros):
    """v1.1: те же правила плюс просьба не выдумывать."""
    return bot_v1(vopros, pravila=PRAVILA_STROGIE)


verno_v1_1 = progon(bot_v1_1, "v1.1: просьба «не выдумывай»")

print(f"Было {verno_v1} из {len(SLUCHAI)}, стало {verno_v1_1} из {len(SLUCHAI)}")

**Что посмотреть в выводе:**

При подготовке лаборатории просьба сработала так: бот ответил **«Не знаю, уточните
у оператора» на все пять вопросов**. Счёт остался прежним: 1 из 5, только теперь
«зелёный» — случай `samokaty`, а угаданная гарантия стала «не знаю».

1. **Выдумки исчезли.** Ни одного неверного факта: это большой шаг, гость больше
   не приедет к закрытой двери.
2. **Знаний не прибавилось.** И не могло: данных у модели по-прежнему нет. Просьба
   может только заставить модель молчать, но не может рассказать ей про «Полярис».
3. **Число не выросло.** Бот стал безопаснее и одновременно бесполезнее: гость,
   которому на любой вопрос отвечают «уточните у оператора», просто позвонит сам.

Это типичная развилка: честный отказ и полезный ответ — разные вещи, и одной строкой
в правилах вторую не получить.

Отсюда вывод, ради которого писался весь модуль: **просьба к модели — не источник
знаний**. Чтобы бот отвечал правильно, факты нужно **положить в запрос**. Этим мы и
займёмся в следующих модулях.

## Шаг 8. Сколько это стоит `[дано]`

Пока считали ответы, мы потратили деньги. Посмотрим сколько — и прикинем масштаб.

In [ ]:
_, usage_primer = sprosit([{"role": "system", "content": PRAVILA_STROGIE},
                           {"role": "user", "content": VOPROSY_GOSTEY[0][1]}], max_tokens=120)

za_vopros = cena(usage_primer)
print(f"Один вопрос: вход {usage_primer.prompt_tokens} токенов, выход {usage_primer.completion_tokens}, "
      f"цена ${za_vopros:.6f}")
for voprosov in (100, 1_000, 10_000, 100_000):
    print(f"   {voprosov:>7} вопросов в месяц: ${za_vopros * voprosov:>8.4f}")

print("\nА теперь то же самое на сильной модели (цены примерно в 100 раз выше):")
for voprosov in (100, 1_000, 10_000, 100_000):
    print(f"   {voprosov:>7} вопросов в месяц: ${za_vopros * voprosov * 100:>8.2f}")

**Что посмотреть в выводе:** при подготовке лаборатории один вопрос стоил $0.000004 —
это 4 цента за десять тысяч вопросов. На сильной модели те же десять тысяч вопросов
обойдутся уже примерно в $4, а сто тысяч — в $40. Это не повод сразу брать дешёвую: сначала нужно, чтобы бот отвечал
**правильно**. Но привычку считать цену заводим с первого дня — в каждой лаборатории
курса есть строка с ценой.

## Шаг 9. Задания `[пиши сам]`

1. **Добавьте свой случай.** Придумайте вопрос гостя, ответ на который есть на сайте
   «Поляриса» (услуги, приём техники, контакты), добавьте его в `SLUCHAI` через
   `zapisat_sluchay` и прогоните обе версии. Стало ли число другим?
2. **Сломайте проверку.** Найдите ответ, который проходит `proverit`, но неверен по сути.
   Подсказка: проверка ищет подстроку. Что будет с ответом «гарантия не 12 месяцев, а 3»?
   Как бы вы это починили?
3. **Температура против фактов.** Прогоните `bot_v1` трижды с `temperature=1.2`
   (передайте её в `sprosit`). Насколько разъехались часы работы в субботу? А теперь
   главный вопрос: спасает ли `temperature=0` от выдумок — или делает их стабильными?
4. **Свой ключ.** Если у вас есть ключ OpenRouter, положите в секреты `AI_BASE_URL`,
   `AI_KEY`, `AI_MODEL` и перезапустите шаги 1 и 6 на другой модели. Насколько отличается
   число верных ответов? Не забудьте поправить `CENA_VHODA` и `CENA_VYHODA`.

## Что записать в файлы курса

**`cases.jsonl`** — уже записан в этом ноутбуке. Скачайте его (панель файлов слева)
и положите в свой проект: это начало эталонного набора, он будет расти весь курс.

**`decisions.md`** — первая запись по рубрике:

> **Что сравнивали.** v1 без правил и v1.1 с просьбой «не выдумывай» на 5 случаях.
> **Числа.** 1 из 5 против 1 из 5. Неверных фактов: было 4, стало 0. Цена не изменилась.
> **Что выбрали.** Просьбу оставляем: неверный факт дороже отказа.
> **Чем пожертвовали.** Полезностью: теперь бот отвечает «уточните у оператора» почти
> на всё. Задача не решена.
> **Когда пересмотреть.** Как только у бота появятся документы: правило придётся
> переписать под них.

**`antipatterns.md`** — «просьба вместо фактов»: почему казалось хорошим, что вышло
на самом деле, когда всё же применимо.

## Что унести с собой

* Модель продолжает текст правдоподобно — поэтому она уверенно **выдумывает** то, чего
  не знает.
* Правила в `system` задают тон и поведение, но **не дают знаний**.
* Провал, который не записан в журнал случаев, вернётся.
* Любое улучшение проверяется числом на одном и том же наборе.
* Цену запроса считают с первого дня, а не когда придёт счёт.